In [25]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import joblib
from sklearn.metrics import precision_recall_curve, auc
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.optimizers import Adam, SGD
from keras.models import load_model
import xgboost as xgb
import lightgbm as lgb
import time

In [27]:
def prepare_data(data):
    X = data.drop(['sample', 'regulator', 'target', 'interaction', 'size'], axis=1).values
    y = data['interaction'].values
    X = np.asarray(X).astype(np.float32)
    y = np.asarray(y).astype(np.float32)
    return X, y

In [7]:
training_set = pd.read_csv(r'./caocao/training/size10.csv', index_col=0)
training_set.shape

(1800000, 107)

In [5]:
def load_size10():
    training_set = pd.read_csv(r'./caocao/training/size10.csv', index_col=0)
    
    # training_set_50_1 = pd.read_csv(r'./caocao/training/size50_10.csv', index_col=0)
    # training_set_50_2 = pd.read_csv(r'./caocao/training/size50_20.csv', index_col=0)
    
    # training_set_50_1_true = training_set_50_1[training_set_50_1['interaction']==1]
    # training_set_50_2_true = training_set_50_2[training_set_50_2['interaction']==1]
    
    # training_set_sum = pd.concat([training_set, training_set_50_1_true, training_set_50_2_true], ignore_index=True)
    # del training_set
    # del training_set_50_1
    # del training_set_50_2
    # del training_set_50_1_true
    # del training_set_50_2_true
    valid_set = pd.read_csv(r'./caocao/validation/size10.csv', index_col=0)
    X_train, y_train = prepare_data(training_set)
    X_valid, y_valid = prepare_data(valid_set)
    # del training_set_sum
    del valid_set
    return X_train, y_train, X_valid, y_valid

In [17]:
def load_size50():
    '''
    Dataset contains all size10, size50 and true positive of size 100
    '''
    #training_set_10 = pd.read_csv(r'./caocao/training/size10.csv', index_col=0)
    training_set_50_1 = pd.read_csv(r'./caocao/training/size50_10.csv', index_col=0)
    training_set_50_2 = pd.read_csv(r'./caocao/training/size50_20.csv', index_col=0)

    # training_set_100_1 = pd.read_csv(r'./caocao/training/size100_5.csv', index_col=0)
    # training_set_100_2 = pd.read_csv(r'./caocao/training/size100_10.csv', index_col=0)
    # training_set_100_3 = pd.read_csv(r'./caocao/training/size100_15.csv', index_col=0)
    # training_set_100_4 = pd.read_csv(r'./caocao/training/size100_20.csv', index_col=0)
    
    # training_set_100_1_true = training_set_100_1[training_set_100_1['interaction']==1]
    # training_set_100_2_true = training_set_100_2[training_set_100_2['interaction']==1]
    # training_set_100_3_true = training_set_100_3[training_set_100_3['interaction']==1]
    # training_set_100_4_true = training_set_100_4[training_set_100_4['interaction']==1]

    
    # training_set_sum = pd.concat([training_set_10, training_set_50_1, training_set_50_2,
    #                              training_set_100_1_true, training_set_100_2_true,
    #                              training_set_100_3_true, training_set_100_4_true], ignore_index=True)

    training_set_sum = pd.concat([training_set_50_1, training_set_50_2], ignore_index=True)
    # del training_set_10
    # del training_set_50_1
    # del training_set_50_2
    # del training_set_100_1_true
    # del training_set_100_2_true
    # del training_set_100_3_true
    # del training_set_100_4_true
    # del training_set_100_1
    # del training_set_100_2
    # del training_set_100_3
    # del training_set_100_4
    training_set_sum = training_set_sum.dropna()
    X_train, y_train = prepare_data(training_set_sum)
    del training_set_sum
    valid_set = pd.read_csv(r'./caocao/validation/size50.csv', index_col=0)
    X_valid, y_valid = prepare_data(valid_set)
    del valid_set
    return X_train, y_train, X_valid, y_valid

In [5]:
def load_size70():
    '''
    Dataset contains all true size10, size50 and all size 70
    '''
    training_set_10 = pd.read_csv(r'./caocao/training/size10.csv', index_col=0)
    training_set_10_true = training_set_10[training_set_10['interaction']==1]
    del training_set_10
    
    training_set_50_1 = pd.read_csv(r'./caocao/training/size50_10.csv', index_col=0)
    training_set_50_2 = pd.read_csv(r'./caocao/training/size50_20.csv', index_col=0)
    training_set_50_1_true = training_set_50_1[training_set_50_1['interaction']==1]
    training_set_50_2_true = training_set_50_2[training_set_50_2['interaction']==1]
    del training_set_50_1
    del training_set_50_2
    
    training_set_70_1 = pd.read_csv(r'./caocao/training/size70_5.csv', index_col=0)
    training_set_70_2 = pd.read_csv(r'./caocao/training/size70_10.csv', index_col=0)
    training_set_70_3 = pd.read_csv(r'./caocao/training/size70_15.csv', index_col=0)
    training_set_70_4 = pd.read_csv(r'./caocao/training/size70_20.csv', index_col=0)

    
    training_set_sum = pd.concat([training_set_10_true, training_set_50_1_true, training_set_50_2_true,
                                 training_set_70_1, training_set_70_2,
                                 training_set_70_3, training_set_70_4], ignore_index=True)

    del training_set_10_true
    del training_set_50_1_true
    del training_set_50_2_true
    del training_set_70_1
    del training_set_70_2
    del training_set_70_3
    del training_set_70_4
    training_set_sum = training_set_sum.dropna()
    
    X_train, y_train = prepare_data(training_set_sum)
    del training_set_sum
    valid_set = pd.read_csv(r'./caocao/validation/size70.csv', index_col=0)
    X_valid, y_valid = prepare_data(valid_set)
    del valid_set
    return X_train, y_train, X_valid, y_valid

In [5]:
def load_size100():
    '''
    Dataset contains all true size10, size50 and all size 100
    '''
    # training_set_10 = pd.read_csv(r'./caocao/training/size10.csv', index_col=0)
    # training_set_10_true = training_set_10[training_set_10['interaction']==1]
    # del training_set_10
    
    # training_set_50_1 = pd.read_csv(r'./caocao/training/size50_10.csv', index_col=0)
    # training_set_50_2 = pd.read_csv(r'./caocao/training/size50_20.csv', index_col=0)
    # training_set_50_1_true = training_set_50_1[training_set_50_1['interaction']==1]
    # training_set_50_2_true = training_set_50_2[training_set_50_2['interaction']==1]
    # del training_set_50_1
    # del training_set_50_2
    
    training_set_100_1 = pd.read_csv(r'./caocao/training/size100_5.csv', index_col=0)
    training_set_100_2 = pd.read_csv(r'./caocao/training/size100_10.csv', index_col=0)
    training_set_100_3 = pd.read_csv(r'./caocao/training/size100_15.csv', index_col=0)
    training_set_100_4 = pd.read_csv(r'./caocao/training/size100_20.csv', index_col=0)

    
    # training_set_sum = pd.concat([training_set_10_true, training_set_50_1_true, training_set_50_2_true,
    #                              training_set_100_1, training_set_100_2,
    #                              training_set_100_3, training_set_100_4], ignore_index=True)
    training_set_sum = pd.concat([training_set_100_1, training_set_100_2,
                                 training_set_100_3, training_set_100_4], ignore_index=True)


    # del training_set_10_true
    # del training_set_50_1_true
    # del training_set_50_2_true
    del training_set_100_1
    del training_set_100_2
    del training_set_100_3
    del training_set_100_4
    training_set_sum = training_set_sum.dropna()
    
    X_train, y_train = prepare_data(training_set_sum)
    del training_set_sum
    valid_set = pd.read_csv(r'./caocao/validation/size100.csv', index_col=0)
    X_valid, y_valid = prepare_data(valid_set)
    del valid_set
    return X_train, y_train, X_valid, y_valid

In [7]:
def get_test_set(size):
    test_set_report = pd.read_csv(fr'./jump3_code/data for comparison/size{size}/size{size}.csv', index_col=0)
    X_test_report, y_test_report = prepare_data(test_set_report)
    del test_set_report
    return X_test_report, y_test_report

In [9]:
def get_report(model,report_filename, size, sample_number_max):
    edges = size*(size-1)
    start_predition_time = time.time()
    y_test_report_pred = model.predict(X_test)
    end_predition_time = time.time()
    with open(report_filename, 'w+') as f:
        for i in range(0, edges*sample_number_max, edges):
            precision1, recall1, _ = precision_recall_curve(y_test[i:i+edges], y_test_report_pred[i:i+edges])
            aupr1 = auc(recall1, precision1)
            f.write(f'{aupr1}\n')         
    return (end_predition_time-start_predition_time)*1000   

In [11]:
def get_report_xgb(model,report_filename, size, sample_number_max):
    edges = size*(size-1)
    dval = xgb.DMatrix(X_test, label=y_test)
    start_predition_time = time.time()
    y_test_report_pred = model.predict(dval)
    end_predition_time = time.time()
    with open(report_filename, 'w+') as f:
        for i in range(0, edges*sample_number_max, edges):
            precision1, recall1, _ = precision_recall_curve(y_test[i:i+edges], y_test_report_pred[i:i+edges])
            aupr1 = auc(recall1, precision1)
            f.write(f'{aupr1}\n')           
    return (end_predition_time-start_predition_time)*1000  

In [13]:
def get_report_lgb(model, report_filename, size, sample_number_max):
    edges = size*(size-1)
    start_predition_time = time.time()
    y_test_report_pred = model.predict(X_test, num_iteration=model.best_iteration)
    end_predition_time = time.time()
    with open(report_filename, 'w+') as f:
        for i in range(0, edges*sample_number_max, edges):
            precision1, recall1, _ = precision_recall_curve(y_test[i:i+edges], y_test_report_pred[i:i+edges])
            aupr1 = auc(recall1, precision1)
            f.write(f'{aupr1}\n')            
    return (end_predition_time-start_predition_time)*1000  

In [19]:
X_train, y_train, X_valid, y_valid = load_size50()

In [21]:
X_test, y_test = get_test_set(50)

**Test with Dream4 dataset**

In [15]:
model = load_model('./caocao/trained_model/model_10.h5')

In [17]:
def loadData():
    dataset = pd.read_csv(r'./caocao/Dream4/size10_1.csv', index_col=0)
    X_test, y_test = prepare_data(dataset)
    return X_test, y_test

In [21]:
def get_report_Dream4(model, X_test, y_test,report_filename, size, sample_number_max):
    edges = size*(size-1)
    start_predition_time = time.time()
    y_test_report_pred = model.predict(X_test)
    end_predition_time = time.time()
    with open(report_filename, 'w+') as f:
        for i in range(0, edges*sample_number_max, edges):
            precision1, recall1, _ = precision_recall_curve(y_test[i:i+edges], y_test_report_pred[i:i+edges])
            aupr1 = auc(recall1, precision1)
            f.write(f'{aupr1}\n')         
    return (end_predition_time-start_predition_time)*1000   

In [23]:
dr4_X_test, dr4_y_test = loadData()
prediction_time = get_report_Dream4(model,dr4_X_test, dr4_y_test, f'./caocao/report_update/dream4/size10_1.txt', 10, 5)
print(prediction_time)

ValueError: Exception encountered when calling Sequential.call().

[1mInput 0 of layer "dense" is incompatible with the layer: expected axis -1 of input shape to have value 102, but received input with shape (32, 42)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(32, 42), dtype=float32)
  • training=False
  • mask=None

**End with Dream4 dataset test**

In [17]:
with open('./caocao/report/times_70.txt', 'w+') as f:
    model_size = 70
    model = keras.models.Sequential([ 
        keras.layers.Dense(1024, activation="relu", input_shape=X_train.shape[1:]),
        keras.layers.BatchNormalization(),
        keras.layers.Dense(512, activation="relu"),
        keras.layers.BatchNormalization(),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.BatchNormalization(),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.BatchNormalization(),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.BatchNormalization(),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.BatchNormalization(),
        keras.layers.Dense(16, activation="relu"),
        keras.layers.BatchNormalization(),
        keras.layers.Dense(8, activation="relu"),
        keras.layers.BatchNormalization(),
        keras.layers.Dense(1, activation="sigmoid")
    ])
    
    #optimizer = keras.optimizers.RMSprop(lr=0.001, rho=0.9)
    optimizer = SGD(clipvalue=1.0, momentum=0.9, nesterov=True)
    #optimizer=Adam(learning_rate=0.01, beta_1=0.9, beta_2=0.999)
    early_stopping_cb = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
    model.compile(loss="mean_squared_error", optimizer=optimizer, metrics=['accuracy'])
    mlp_start_training_time = time.time()
    history = model.fit(X_train, y_train, epochs=100, validation_data=(X_valid, y_valid), 
                        callbacks=[early_stopping_cb])
    model.save('./caocao/trained_model/model_70.h5')
    #model = load_model('./caocao/trained_model/model_70.h5')
    mlp_end_training_time = time.time()
    f.write(f'{(mlp_end_training_time - mlp_start_training_time)*1000}\n')
    print(f'{(mlp_end_training_time - mlp_start_training_time)*1000}\n')
    #model.save(fr'./caocao/trained_model/size{model_size}_mlp.h5')
    #mlp_prediction_time = get_report(model, f'./caocao/report_update/model{model_size}/mlp_size50.txt', 50, 4)
    #mlp_prediction_time = get_report(model, f'./caocao/report_update/model{model_size}/mlp_size10.txt', 10, 10)
    mlp_prediction_time = get_report(model, f'./caocao/report_update/model{model_size}/mlp_size70.txt', model_size, 8)
    f.write(f'{mlp_prediction_time}\n')
    del model
    ##############################################################
    lin_reg = LinearRegression()
    lr_start_training_time = time.time()
    lin_reg.fit(X_train, y_train)
    lr_end_training_time = time.time()
    f.write(f'{(lr_end_training_time - lr_start_training_time)*1000}\n')
    #joblib.dump(lin_reg, fr'./caocao/trained_model/size{model_size}_lr.pkl')

   # lr_prediction_time = get_report(lin_reg, f'./caocao/report_update/model{model_size}/linear_size10.txt', 10, 10)
   # lr_prediction_time = get_report(lin_reg, f'./caocao/report_update/model{model_size}/linear_size50.txt', 50, 4)
    lr_prediction_time = get_report(lin_reg, f'./caocao/report_update/model{model_size}/linear_size70.txt', model_size, 8)

    f.write(f'{lr_prediction_time}\n')
    del lin_reg
    ##############################################################
    dt_model = DecisionTreeClassifier(random_state=42)
    dt_start_training_time = time.time()
    dt_model.fit(X_train, y_train)
    dt_end_training_time = time.time()
    f.write(f'{(dt_end_training_time - dt_start_training_time)*1000}\n')
   # joblib.dump(dt_model, fr'./caocao/trained_model/size{model_size}_dt.pkl')
    #dt_prediction_time = get_report(dt_model, f'./caocao/report_update/model{model_size}/dt_size50.txt', 50, 4)
    #dt_prediction_time = get_report(dt_model, f'./caocao/report_update/model{model_size}/dt_size10.txt', 10, 10)
    dt_prediction_time = get_report(dt_model, f'./caocao/report_update/model{model_size}/dt_size70.txt', model_size, 8)
    f.write(f'{dt_prediction_time}\n')
    del dt_model
    ##############################################################
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_start_training_time = time.time()
    rf_model.fit(X_train, y_train)
    rf_end_training_time = time.time()
    f.write(f'{(rf_end_training_time - rf_start_training_time)*1000}\n')
    #joblib.dump(rf_model, fr'./caocao/trained_model/size{model_size}_rf.pkl')
    #rf_prediction_time = get_report(rf_model, f'./caocao/report_update/model{model_size}/rf_size50.txt', 50, 4)
    #rf_prediction_time = get_report(rf_model, f'./caocao/report_update/model{model_size}/rf_size10.txt', 10, 10)
    rf_prediction_time = get_report(rf_model, f'./caocao/report_update/model{model_size}/rf_size70.txt', model_size, 8)
    f.write(f'{rf_prediction_time}\n')
    del rf_model
    ##############################################################
    knn_model = KNeighborsClassifier(n_neighbors=10)
    # k=10 is the best
    # Train the model
    knn_start_training_time = time.time()
    knn_model.fit(X_train, y_train)
    knn_end_training_time = time.time()
    f.write(f'{(knn_end_training_time - knn_start_training_time)*1000}\n')
    #joblib.dump(knn_model, fr'./caocao/trained_model/size{model_size}_knn.pkl')
    #knn_prediction_time = get_report(knn_model, f'./caocao/report_update/model{model_size}/knn_size50.txt', 50, 4)
    #knn_prediction_time = get_report(knn_model, f'./caocao/report_update/model{model_size}/knn_size10.txt', 10, 10)
    knn_prediction_time = get_report(knn_model, f'./caocao/report_update/model{model_size}/knn_size70.txt', model_size, 8)
    f.write(f'{knn_prediction_time}\n')
    del knn_model
    ##############################################################
    
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval = xgb.DMatrix(X_valid, label=y_valid)
    params = {
        'max_depth': 3,         # Maximum depth of a tree
        'eta': 0.1,             # Learning rate
        'objective': 'binary:logistic',  # Binary classification objective
        'eval_metric': 'logloss' # Evaluation metric
    }
    evallist = [(dtrain, 'train'), (dval, 'eval')]
    num_round = 100  # Number of boosting rounds
    xgb_start_training_time = time.time()
    xgb_model = xgb.train(params, dtrain, num_round, evals=evallist, early_stopping_rounds=10)
    xgb_end_training_time = time.time()
    f.write(f'{(xgb_end_training_time - xgb_start_training_time)*1000}\n')
    #xgb_prediction_time = xgb_model.save_model(fr'./caocao/trained_model/size{model_size}_xgb.pkl')
    #xgb_prediction_time = get_report_xgb(xgb_model, f'./caocao/report_update/model{model_size}/xgb_size10.txt', 10, 10)    
    #xgb_prediction_time = get_report_xgb(xgb_model, f'./caocao/report_update/model{model_size}/xgb_size50.txt', 50, 4)
    xgb_prediction_time = get_report_xgb(xgb_model, f'./caocao/report_update/model{model_size}/xgb_size70.txt', model_size, 8)
    f.write(f'{xgb_prediction_time}\n')
    del xgb_model
    ##############################################################
    train_data = lgb.Dataset(X_train, label=y_train)
    val_data = lgb.Dataset(X_valid, label=y_valid, reference=train_data)
    params = {
        'boosting_type': 'gbdt',  # Gradient Boosting Decision Tree
        'objective': 'binary',    # Binary classification
        'metric': 'binary_logloss', # Metric to evaluate
        'num_leaves': 31,         # Maximum tree leaves for base learners
        'learning_rate': 0.05,    # Learning rate
        'feature_fraction': 0.9   # Fraction of features to be used for each tree
    }
    lgb_start_training_time = time.time()
    lgb_model = lgb.train(params, train_data, num_boost_round=100, valid_sets=[train_data, val_data])
    lgb_end_training_time = time.time()
    f.write(f'{(lgb_end_training_time - lgb_start_training_time)*1000}\n')
   # lgb_model.save_model(fr'./caocao/trained_model/size{model_size}_lgb.txt')
   # lgb_prediction_time = get_report_lgb(lgb_model, f'./caocao/report_update/model{model_size}/lgb_size10.txt', 10, 10)
    #lgb_prediction_time = get_report_lgb(lgb_model, f'./caocao/report_update/model{model_size}/lgb_size50.txt', 50, 4)
    lgb_prediction_time = get_report_lgb(lgb_model, f'./caocao/report_update/model{model_size}/lgb_size70.txt', model_size, 8)
    f.write(f'{lgb_prediction_time}\n')
    del lgb_model
    ##############################################################
    nb_classifier = GaussianNB()
    nb_start_training_time = time.time()
    nb_classifier.fit(X_train, y_train)
    nb_end_training_time = time.time()
    f.write(f'{(nb_end_training_time - nb_start_training_time)*1000}\n')
    #nb_prediction_time = get_report(nb_classifier, f'./caocao/report_update/model{model_size}/nb_size50.txt', 50, 4)
    #nb_prediction_time = get_report(nb_classifier, f'./caocao/report_update/model{model_size}/nb_size10.txt', 10, 10)
    nb_prediction_time = get_report(nb_classifier, f'./caocao/report_update/model{model_size}/nb_size70.txt', model_size, 8)
    f.write(f'{nb_prediction_time}\n')
    del nb_classifier

/home/cao/anaconda3/lib/python3.12/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100
 31267/316754 ━━━━━━━━━━━━━━━━━━━━ 45:35 10ms/step - accuracy: 0.9230 - loss: 0.0671

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



 61445/316754 ━━━━━━━━━━━━━━━━━━━━ 40:47 10ms/step - accuracy: 0.9248 - loss: 0.0656

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



...

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



316754/316754 ━━━━━━━━━━━━━━━━━━━━ 3250s 10ms/step - accuracy: 0.9311 - loss: 0.0602 - val_accuracy: 0.9653 - val_loss: 0.0335
Epoch 15/100
 17950/316754 ━━━━━━━━━━━━━━━━━━━━ 47:48 10ms/step - accuracy: 0.9320 - loss: 0.0595

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



316754/316754 ━━━━━━━━━━━━━━━━━━━━ 3254s 10ms/step - accuracy: 0.9311 - loss: 0.0602 - val_accuracy: 0.9652 - val_loss: 0.0335


48786001.32584572

1208/1208 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step
[0]	train-logloss:0.29160	eval-logloss:0.21621
[1]	train-logloss:0.28557	eval-logloss:0.20890
[2]	train-logloss:0.28027	eval-logloss:0.20243
[3]	train-logloss:0.27569	eval-logloss:0.19677
[4]	train-logloss:0.27171	eval-logloss:0.19171
[5]	train-logloss:0.26820	eval-logloss:0.18727
[6]	train-logloss:0.26512	eval-logloss:0.18328
[7]	train-logloss:0.26250	eval-logloss:0.17982
[8]	train-logloss:0.26018	eval-logloss:0.17674
[9]	train-logloss:0.25810	eval-logloss:0.17393
[10]	train-logloss:0.25624	eval-logloss:0.17142
[11]	train-logloss:0.25466	eval-logloss:0.16928
[12]	train-logloss:0.25318	eval-logloss:0.16729
[13]	train-logloss:0.25191	eval-logloss:0.16552
[14]	train-logloss:0.25079	eval-logloss:0.16403
[15]	train-logloss:0.24978	eval-logloss:0.16262
[16]	train-logloss:0.24885	eval-logloss:0.16143
[17]	train-logloss:0.24799	eval-logloss:0.16031
[18]	train-logloss:0.24722	eval-logloss:0.15934
[19]	train-logloss:0.24655	eval-logl

In [ ]:
X_train, y_train, X_valid, y_valid = load_size100()

In [31]:
nb_classifier = GaussianNB()
nb_start_training_time = time.time()
nb_classifier.fit(X_train, y_train)
nb_end_training_time = time.time()
#nb_prediction_time = get_report(nb_classifier, f'./caocao/nb_size100.txt', 100, 4)
#nb_prediction_time = get_report(nb_classifier, f'./caocao/nb_size50.txt', 50, 4)
#nb_prediction_time = get_report(nb_classifier, f'./caocao/nb_size10.txt', 10, 10)
#nb_prediction_time = get_report(nb_classifier, f'./caocao/report_update/model{model_size}/nb_size70.txt', model_size, 8)
print(f'{(nb_end_training_time - nb_start_training_time)*1000}\n')

3705.824136734009



In [29]:
X_test, y_test = get_test_set(100)

In [23]:
nb_prediction_time = get_report(nb_classifier, f'./caocao/nb_size10.txt', 10, 10)

In [27]:
nb_prediction_time = get_report(nb_classifier, f'./caocao/nb_size50.txt', 50, 4)

In [31]:
nb_prediction_time = get_report(nb_classifier, f'./caocao/nb_size100.txt', 100, 4)